# 03 — Final Transformer TEST evaluation

Run after **01 patient-level split** and **02 Transformer training**. This notebook loads the completed checkpoint selected by maximum VALIDATION average precision, checks the exact tensor/split/feature order, scores the untouched TEST split and produces aggregate evaluation artifacts.

Inputs are the existing private bundle, frozen full snapshot manifest and selected checkpoint, configured by `TAK861_CONFIG` or `config.local.json`. Outputs include AP, ROC-AUC, precision, recall, F1, a confusion matrix, decile/gains/lift tables, response rates by decile, and Recall/Precision@10%, 20% and 30%. There are no sample metrics in this repository.

**Freeze the model choice before this notebook.** No training, threshold selection or calibration occurs here. The threshold was selected on VALIDATION in 02. LightGBM comparison remains deferred. Repeated snapshots count separately; this is snapshot targeting, not unique-patient outreach.

In [ ]:
from pathlib import Path
import sys
import os

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "targeting_evaluation.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the cloned repository or a notebook directory inside it.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display
from src.configuration import load_config
from src.data_utils import load_bundle, validate_population, validate_source_review
from src.evaluation import evaluate_checkpoint
from targeting_evaluation import read_input


## Load frozen inputs

Paths are resolved relative to the local JSON configuration. The bundle loader checks the saved input fingerprint and source-review declarations. These declarations document the upstream review; tensor values cannot independently prove clinical cutoff or observability logic. The report directory must be new or empty to prevent overwriting a previous evaluation.

In [ ]:
cfg = load_config(os.environ.get("TAK861_CONFIG", ROOT / "config.local.json"))
validate_source_review(cfg["source_review"])
bundle = load_bundle(cfg["bundle_dir"])
validate_population(bundle, cfg["expected"])
manifest = read_input(cfg["manifest_path"])
checkpoint_path = cfg["run_dir"] / "best_transformer.pt"
if not checkpoint_path.is_file():
    raise FileNotFoundError("Complete notebook 02 and select its final checkpoint before TEST evaluation.")


## Score TEST once and apply the fixed VALIDATION threshold

The scorer carries `PATIENT_ID + END_DT` from the same indexed records as each prediction. It rejects changed labels, keys, tensor content, feature order, timesteps or split assignments. Model evaluation mode and inference mode are used, and sigmoid is applied once to each output logit.

No patient-level prediction file is exported. Only aggregate tables/charts and run fingerprints are saved. Keep even aggregate run outputs in the approved work environment.

In [ ]:
evaluation = evaluate_checkpoint(
    bundle, manifest, checkpoint_path, cfg["report_dir"],
    device=cfg["training"].get("device", "auto"),
)
results = evaluation["results"]
display(evaluation["classification_metrics"])


## Decile table

Sort TEST snapshots by predicted RESP=1 probability, highest first, with a fixed label-independent snapshot-key hash for ties. For `N` snapshots, cumulative boundary `d` is `ceil(d*N/10)`. Decile 1 is highest propensity; decile 10 is lowest. Labels are used only after ranking to count observed responses.

`response_rate` and `decile_precision` describe one decile. `cumulative_precision` describes all snapshots selected through that decile. The proportion of all positives captured in a decile and cumulative capture/recall use **all TEST positives** as denominator. Lift uses the **overall TEST response rate**, not training prevalence.

In [ ]:
def format_aggregate_table(frame):
    shown = frame.copy()
    rates = {"population_fraction", "response_rate", "decile_precision", "share_of_all_resp1",
             "cumulative_population_fraction", "cumulative_recall", "cumulative_precision",
             "overall_test_response_rate", "actual_population_fraction", "recall", "precision"}
    for column in rates.intersection(shown.columns):
        shown[column] = shown[column].map(lambda x: "N/A" if pd.isna(x) else f"{x:.2%}")
    for column in {"decile_lift", "cumulative_lift", "lift"}.intersection(shown.columns):
        shown[column] = shown[column].map(lambda x: "N/A" if pd.isna(x) else f"{x:.3f}x")
    return shown

display(format_aggregate_table(results["deciles"]))


## Top 10%, 20% and 30%

These are cumulative deciles 1, 1–2 and 1–3, so the tables and chart endpoints agree exactly. Integer rounding can slightly exceed the nominal selected percentage; both selected counts and actual coverage are included. Top-decile lift is `Precision@10% / TEST prevalence`. No TEST label is used to choose these capacities.

In [ ]:
display(format_aggregate_table(results["topk"]))
lift = results["deciles"].iloc[0]["decile_lift"]
display(Markdown("**Top-decile lift:** " + ("N/A" if pd.isna(lift) else f"{lift:.3f}x")))
display(results["ties"])


## Charts and report

AP is the non-interpolated average-precision summary of the precision–recall curve; it is not trapezoidal PR-AUC. Global ROC-AUC is supporting context. The gains, lift and top-K panels directly show how concentrated the positive snapshots are in the highest-scored populations.

Class-weighted BCE sigmoid scores are not guaranteed to be calibrated probabilities. Ties can limit targeting resolution. Snapshot dependence is not captured by confidence intervals in this run; none are computed. No incremental claim versus LightGBM can be made until it is evaluated separately on this exact frozen TEST population.

In [ ]:
for name in ("gains", "lift", "topk_performance", "response_rate", "discrimination", "confusion_matrix"):
    display(Image(filename=str(evaluation["paths"][name])))

display(Markdown("**Saved aggregate files:**\n\n" + "\n".join(
    "- `" + Path(path).name + "`" for path in evaluation["paths"].values()
)))


The Markdown report combines the results and interpretation. Clear notebook outputs before committing or sharing source. Do not use this TEST report to adjust features, model settings, calibration or the operating threshold; any subsequent iteration needs a separately governed evaluation design.